In [2]:
import json
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional


In [3]:
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

In [4]:
def call_llm(system_prompt, user_query, model_name="Qwen/Qwen2.5-14B-Instruct" , response_model=None):
    api_params = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content

In [8]:
class EntityRelation(BaseModel):
    source_entity: str = Field(description="The first entity in the relationship.")
    target_entity: str = Field(description="The second entity in the relationship.")
    relation: str = Field(
        description="Concise relation in snake_case (1-3 words). Use NONE if no direct relation. Before producing JSON, internally form a sentence: Entity 1 [relation] Entity 2. If this sentence is not valid or not supported by the text, return NONE."
    )

class SentenceRelations(BaseModel):
    # 🔥 CRITICAL: The scratchpad MUST be the first field.
    step_by_step_reasoning: str = Field(
        description="Think step-by-step. Read the sentence and the list of entities. Identify which entities interact or share a direct relationship, and explain why before listing them."
    )
    relationships: List[EntityRelation] = Field(
        description="A list of all identified relationships between the provided entities. If no entities are related, return an empty list.",
        default_factory=list
    )

In [12]:
with open("./entity_output_files/chapter_0_output.json", "r", encoding='utf-8') as f:
    chapter = json.load(f)
    print(len(chapter))
    for group in chapter:
        print(len(group['extracted_entities']))
        for data in group['extracted_entities'][:10]:
            print(data['sentence'], data['entities'])
            with open("prompts/relation_extraction.txt", 'r', encoding='utf-8') as f:
                relation_extraction_prompt = f.read()
            sentence = f"sentence: {data['sentence']}\n"
            for i in range(len(data['entities'])):
                for j in range(i+1, len(data['entities'])):
                    entities = f"entity 1: {data['entities'][i]}\nentity 2: {data['entities'][j]}\n"
                    print(sentence+entities)
                    response = call_llm(relation_extraction_prompt, sentence+entities, response_model=SentenceRelations)
                    print(response)
                    print("----------------------------------")
        break

5
50
संवत्सरारम्भ के लिए सूर्योदय व्यापिनी प्रतिपदा लेनी चाहिए। ['संवत्सरारम्भ', 'सूर्योदय', 'प्रतिपदा']
sentence: संवत्सरारम्भ के लिए सूर्योदय व्यापिनी प्रतिपदा लेनी चाहिए।
entity 1: संवत्सरारम्भ
entity 2: सूर्योदय

{
  "step_by_step_reasoning": "The sentence states 'For the New Year, the sunrise inclusive Pratipada should be taken.' The relation between संवत्सरारम्भ (New Year) and सूर्योदय (sunrise) is indirect as it involves प्रतिपदा (Pratipada). Hence, the relation is not direct.",
  "relationships": [
    {
      "source_entity": "संवत्सरारम्भ",
      "target_entity": "सूर्योदय",
      "relation": "NONE"
    }
  ]
}
----------------------------------
sentence: संवत्सरारम्भ के लिए सूर्योदय व्यापिनी प्रतिपदा लेनी चाहिए।
entity 1: संवत्सरारम्भ
entity 2: प्रतिपदा

{
  "step_by_step_reasoning": "The sentence states that for the संवत्सरारम्भ (New Year), the प्रतिपदा (first day) with sunrise should be taken into account. This indicates that the New Year is determined based on the first d

In [3]:
# %%
import json
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional


# %%
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')


# %%
def call_llm(system_prompt, user_query, model_name="Qwen/Qwen2.5-14B-Instruct", response_model=None):
    api_params = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 512,   # 🔻 reduced (no reasoning needed)
        "temperature": 0.1,  # 🔻 more deterministic
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# %%
class EntityRelation(BaseModel):
    source_entity: str = Field(
        description="Must EXACTLY match Entity 1"
    )
    target_entity: str = Field(
        description="Must EXACTLY match Entity 2"
    )
    relation: str = Field(
        description="Concise snake_case relation (1-3 words) OR 'NONE'"
    )


# %%
with open("./entity_output_files/chapter_0_output.json", "r", encoding='utf-8') as f:
    chapter = json.load(f)

    for x, group in enumerate(chapter):
        for y, data in enumerate(group['extracted_entities'][:10]):

            sentence = f"Sentence:\n{data['sentence']}\n"
            print(f"group {x}, sentencce {y}")
            for i in range(len(data['entities'])):
                for j in range(i + 1, len(data['entities'])):

                    entity1 = data['entities'][i]
                    entity2 = data['entities'][j]

                    user_query = (
                        f"{sentence}"
                        f"Entity 1: {entity1}\n"
                        f"Entity 2: {entity2}\n"
                    )
                    print(user_query)

                    with open("prompts/relation_extraction.txt", 'r', encoding='utf-8') as f:
                        system_prompt = f.read()

                    response = call_llm(
                        system_prompt,
                        user_query,
                        response_model=EntityRelation
                    )

                    print(response)
                    print("----------------------------------")
                    
                    user_query = (
                        f"{sentence}"
                        f"Entity 1: {entity2}\n"
                        f"Entity 2: {entity1}\n"
                    )
                    print(user_query)

                    with open("prompts/relation_extraction.txt", 'r', encoding='utf-8') as f:
                        system_prompt = f.read()

                    response = call_llm(
                        system_prompt,
                        user_query,
                        response_model=EntityRelation
                    )

                    print(response)
                    print("----------------------------------")

        break

group 0, sentencce 0
Sentence:
संवत्सरारम्भ के लिए सूर्योदय व्यापिनी प्रतिपदा लेनी चाहिए।
Entity 1: संवत्सरारम्भ
Entity 2: सूर्योदय

{
  "source_entity": "संवत्सरारम्भ",
  "target_entity": "सूर्योदय",
  "relation": "NONE"
}
----------------------------------
Sentence:
संवत्सरारम्भ के लिए सूर्योदय व्यापिनी प्रतिपदा लेनी चाहिए।
Entity 1: सूर्योदय
Entity 2: संवत्सरारम्भ

{
  "source_entity": "संवत्सरारम्भ",
  "target_entity": "सूर्योदय",
  "relation": "determined_by"
}
----------------------------------
Sentence:
संवत्सरारम्भ के लिए सूर्योदय व्यापिनी प्रतिपदा लेनी चाहिए।
Entity 1: संवत्सरारम्भ
Entity 2: प्रतिपदा

{
  "source_entity": "संवत्सरारम्भ",
  "target_entity": "प्रतिपदा",
  "relation": "determined_by"
}
----------------------------------
Sentence:
संवत्सरारम्भ के लिए सूर्योदय व्यापिनी प्रतिपदा लेनी चाहिए।
Entity 1: प्रतिपदा
Entity 2: संवत्सरारम्भ

{
  "source_entity": "प्रतिपदा",
  "target_entity": "संवत्सरारम्भ",
  "relation": "determined_by"
}
----------------------------------
